In [1]:
!git clone https://github.com/ayiii-a/Detecting-LLM-Generated-Social-Media-Accounts-Using-Semantic-and-Textual.git

Cloning into 'Detecting-LLM-Generated-Social-Media-Accounts-Using-Semantic-and-Textual'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 13 (delta 1), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 3.77 MiB | 9.59 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import os
%cd Detecting-LLM-Generated-Social-Media-Accounts-Using-Semantic-and-Textual

/content/Detecting-LLM-Generated-Social-Media-Accounts-Using-Semantic-and-Textual


In [3]:
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size=100, hidden_size=128):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # batch_first=True handel(Batch, Seq_Len)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        # Human=0, AI=1
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        # x: (Batch, Seq_Len)
        embeds = self.embedding(x)
        # lstm_out: (Batch, Seq_Len, Hidden_Size)
        # h_n: (1, Batch, Hidden_Size)
        _, (h_n, _) = self.lstm(embeds)

        logits = self.fc(h_n.squeeze(0))
        return logits

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. read data
training_dataset = pd.read_csv("./data/data_for_preprocessing.csv", index_col=0)

# 2. map and extract features
label_mapping = {"human": 0, "ai": 1}
training_dataset['label'] = training_dataset['Author'].str.lower().map(label_mapping)

# 3. separate data
X_train, X_test, y_train, y_test = train_test_split(
    training_dataset['Text'],
    training_dataset['label'],
    test_size=0.2,
    random_state=42,
    stratify=training_dataset['label']
)

print(f"X_train shape: {X_train.shape}")

X_train shape: (4855,)


In [5]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, TensorDataset
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size

def preprocess_for_lstm(texts, tokenizer, max_len=256):
    # Hugging Face
    encoded = tokenizer(
        texts.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )
    return encoded['input_ids']

# Tensor
X_train_tensor = preprocess_for_lstm(X_train, tokenizer)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)

print("DataLoader ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DataLoader ready


In [6]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
import pandas as pd

# 1. Initialize Model, Loss Function, and Optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize model using vocab_size defined in the Tokenizer cell
model = LSTMClassifier(vocab_size=vocab_size, embed_size=100, hidden_size=128).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 2. Training Loop
epochs = 5
print("\n--- Starting Training ---")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(texts) # outputs shape: (Batch, 2)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("Training complete!\n")

# 3. Cross-Dataset Evaluation
print("--- Evaluating on Unseen Dataset ---")

# Load the independent evaluation dataset
test_dataset = pd.read_csv("./data/AI_Generated_Essays_Dataset.csv")
df_eval = test_dataset.dropna(subset=['text'])
y_eval_true = df_eval['generated'].values

# Preprocess the evaluation set (must use the exact same tokenizer and max_len as training)
X_eval_tensor = preprocess_for_lstm(df_eval['text'], tokenizer, max_len=256)

# Create DataLoader for the evaluation set, shuffle=False is critical for accurate metric alignment
y_eval_tensor = torch.tensor(y_eval_true, dtype=torch.long)
eval_loader = DataLoader(TensorDataset(X_eval_tensor, y_eval_tensor), batch_size=32, shuffle=False)

# Prediction Phase
model.eval()
all_preds = []
all_probs = []

with torch.no_grad():
    for texts, _ in eval_loader:
        texts = texts.to(device)
        outputs = model(texts) # Raw logits

        # Convert logits to probabilities (required for AUC-ROC calculation)
        probs = torch.softmax(outputs, dim=1)[:, 1] # Extract probability for class 1 (AI)

        # Extract the predicted class (0 or 1)
        preds = torch.argmax(outputs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate and print evaluation metrics
f1_eval = f1_score(y_eval_true, all_preds)
auc_roc_eval = roc_auc_score(y_eval_true, all_probs)

print(f"\n--- Cross-Dataset Evaluation Metrics (LSTM) ---")
print(f"F1-Score: {f1_eval:.4f} (Baseline TF-IDF was ~0.15)")
print(f"AUC-ROC:  {auc_roc_eval:.4f}\n")

print("--- Detailed Classification Report ---")
print(classification_report(y_eval_true, all_preds, target_names=['Human (0)', 'AI (1)']))

print("--- Confusion Matrix ---")
cm_eval = confusion_matrix(y_eval_true, all_preds)
print(f"True Negatives (Human correctly identified): {cm_eval[0][0]}")
print(f"False Positives (Human misclassified as AI): {cm_eval[0][1]}")
print(f"False Negatives (AI misclassified as Human): {cm_eval[1][0]}")
print(f"True Positives (AI correctly identified):    {cm_eval[1][1]}")

Using device: cpu

--- Starting Training ---
Epoch [1/5], Loss: 0.2026
Epoch [2/5], Loss: 0.0467
Epoch [3/5], Loss: 0.0367
Epoch [4/5], Loss: 0.0311
Epoch [5/5], Loss: 0.0293
Training complete!

--- Evaluating on Unseen Dataset ---

--- Cross-Dataset Evaluation Metrics (LSTM) ---
F1-Score: 0.3810 (Baseline TF-IDF was ~0.15)
AUC-ROC:  0.7584

--- Detailed Classification Report ---
              precision    recall  f1-score   support

   Human (0)       0.95      1.00      0.98      1375
      AI (1)       1.00      0.24      0.38        85

    accuracy                           0.96      1460
   macro avg       0.98      0.62      0.68      1460
weighted avg       0.96      0.96      0.94      1460

--- Confusion Matrix ---
True Negatives (Human correctly identified): 1375
False Positives (Human misclassified as AI): 0
False Negatives (AI misclassified as Human): 65
True Positives (AI correctly identified):    20
